Importing libraries

In [1]:
%pip install torch pandas scikit-learn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from typing import Tuple, Optional, Any


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Load and Explore Dataset

In [2]:
# Read the electricity dataset from CSV into a pandas DataFrame
df = pd.read_csv("source/nsp_electricity_dataset.csv")

# Display the dimensions (rows, columns) and the column headers
print(f"Dataset Shape: {df.shape}")
print(f"Columns: {df.columns}")

# Show the first 5 rows to understand the structure
df.head()


Dataset Shape: (438360, 27)
Columns: Index(['timestamp', 'region', 'hour', 'day_of_week', 'month', 'year', 'week',
       'is_weekend', 'season', 'is_holiday', 'temperature_c', 'feels_like_c',
       'humidity_pct', 'wind_speed_kmh', 'precipitation_mm', 'cloud_cover_pct',
       'renewable_pct', 'consumption_kwh', 'price_per_kwh', 'grid_load_pct',
       'co2_kg', 'customer_type', 'demand_response', 'power_outage',
       'anomaly_flag', 'anomaly_type', 'peak_demand_flag'],
      dtype='str')


,timestamp,region,hour,day_of_week,month,year,week,is_weekend,season,is_holiday,...,consumption_kwh,price_per_kwh,grid_load_pct,co2_kg,customer_type,demand_response,power_outage,anomaly_flag,anomaly_type,peak_demand_flag
0,1/1/2015 0:00,Annapolis Valley,0,3,1,2015,1,0,Winter,1,...,125.53,0.1265,51.6,81.94,Mixed,0,0,0,Normal,0
1,1/1/2015 0:00,Cape Breton,0,3,1,2015,1,0,Winter,1,...,207.38,0.1073,53.3,144.88,Mixed,0,0,0,Normal,0
2,1/1/2015 0:00,Halifax,0,3,1,2015,1,0,Winter,1,...,455.58,0.1336,41.5,288.03,Mixed,0,0,0,Normal,0
3,1/1/2015 0:00,Pictou County,0,3,1,2015,1,0,Winter,1,...,69.31,0.1031,43.3,42.40,Mixed,0,0,0,Normal,0
4,1/1/2015 0:00,South Shore,0,3,1,2015,1,0,Winter,1,...,87.77,0.1283,47.9,50.96,Mixed,0,0,0,Normal,0


Target distribution

In [3]:
# Check the distribution of our target variable (anomaly vs normal)
print("Target Variable Distribution (Anomaly Flag):")
print(df["anomaly_flag"].value_counts())


Target Variable Distribution (Anomaly Flag):
anomaly_flag
0    429307
1      9053
Name: count, dtype: int64


Data Preprocessing

In [4]:
# Define the target variable (y). We convert it to a NumPy array for modeling
y: np.ndarray = df["anomaly_flag"].values

# Define the features (X). We drop the target variable, 
# 'anomaly_type' (which leaks information about the target!), 
# and 'timestamp' (which is highly cardinal/unique per row).
X_df: pd.DataFrame = df.drop(columns=["anomaly_flag", "anomaly_type", "timestamp"])

# One-hot encode categorical features (e.g., region, season) into numeric indicator columns
X_encoded: pd.DataFrame = pd.get_dummies(X_df)

# Convert the resulting pandas DataFrame to a raw numpy array for faster mathematical operations
X: np.ndarray = X_encoded.values


Normalize features

In [5]:
# Initialize a Standard Scaler to normalize features to mean=0 and variance=1.
# This ensures that features with large ranges don't dominate the learning process.
scaler: StandardScaler = StandardScaler()

# Fit the scaler on X and immediately transform X
X = scaler.fit_transform(X)


Train/Test Split (80/20)

In [6]:
# Split the dataset into a training set (80%) and a testing set (20%)
# random_state ensures reproducibility across different runs
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


Logistic Regression

In [7]:
# Instantiate a baseline Sklearn Logistic Regression model
# max_iter is increased to 1000 to ensure convergence
model: LogisticRegression = LogisticRegression(max_iter=1000)

# Train the model. .ravel() converts the y_train column vector into a 1D array
model.fit(X_train, y_train.ravel())

# Predict anomalies on the unseen test set
y_pred: np.ndarray = model.predict(X_test)

# Calculate accuracy metrics
acc: float = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Baseline Accuracy: {acc:.4f}")


Logistic Regression Baseline Accuracy: 0.9833


Neural Network From Scratch

* 1 layer

* neurons = number of features

* sigmoid activation

* gradient descent


Sigmoid Function

In [8]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    """
    Computes the sigmoid activation function.
    Maps any real-valued number into the range (0, 1), 
    which can be interpreted as a probability.
    
    Args:
        z (np.ndarray): The linear combination (W*X + b)
        
    Returns:
        np.ndarray: Probabilities between 0 and 1.
    """
    return 1 / (1 + np.exp(-z))


Derivative

In [9]:
def sigmoid_derivative(z: np.ndarray) -> np.ndarray:
    """
    Computes the derivative of the sigmoid function, used during backpropagation.
    """
    s: np.ndarray = sigmoid(z)
    return s * (1 - s)


Initialize Weights

In [10]:
# Set random seed for completely reproducible weight initialization
np.random.seed(42)

# Number of features (input dimensions) is determined by the training data columns
n_features: int = X_train.shape[1]

# Initialize Weights (W) randomly from a normal distribution and scale down by 0.01 
# to keep initial gradients from exploding or vanishing.
# Shape: (features, 1) because we are predicting a single output probability
W: np.ndarray = np.random.randn(n_features, 1) * 0.01

# Initialize Bias (b) to zeros. Shape: (1, 1)
b: np.ndarray = np.zeros((1, 1))

# Hyperparameters for Custom Neural Network training
learning_rate: float = 0.01
epochs: int = 200


Training Loop (Forward + Backprop)

In [11]:
# Ensure y_train is specifically a column vector so its shape exactly matches 
# the model output (A) for proper vector subtraction.
y_train = y_train.reshape(-1, 1)

# Training loop: iterate over the dataset `epochs` times
for epoch in range(epochs):

    # --- FORWARD PASS ---
    # 1. Compute the linear combination of inputs and weights
    # Z shape: (batch_size, 1)
    Z: np.ndarray = np.dot(X_train, W) + b
    
    # 2. Apply the sigmoid non-linearity to obtain raw probabilities
    # A shape: (batch_size, 1)
    A: np.ndarray = sigmoid(Z)

    # --- BACKWARD PASS (Gradient Descent) ---
    # 3. Calculate the difference between predictions and actual labels
    dZ: np.ndarray = A - y_train
    
    # 4. Compute the gradient of the Loss with respect to Weights (W)
    # Average across all training examples by dividing by X_train.shape[0]
    dW: np.ndarray = np.dot(X_train.T, dZ) / X_train.shape[0]
    
    # 5. Compute the gradient of the Loss with respect to Bias (b)
    db: float = np.sum(dZ) / X_train.shape[0]

    # --- UPDATE WEIGHTS ---
    # Shift parameters in the opposite direction of the gradient 
    # to minimize the loss function
    W = W - learning_rate * dW
    b = b - learning_rate * db

    # Log progress every 20 epochs
    if epoch % 20 == 0:
        # Compute binary cross-entropy loss
        # Added 1e-8 inside log() to prevent mathematically undefined np.log(0)
        loss: float = -np.mean(y_train * np.log(A + 1e-8) + (1 - y_train) * np.log(1 - A + 1e-8))
        print(f"Epoch {epoch:03d} | Binary Cross-Entropy Loss: {loss:.4f}")


Epoch 000 | Binary Cross-Entropy Loss: 0.6936
Epoch 020 | Binary Cross-Entropy Loss: 0.6496
Epoch 040 | Binary Cross-Entropy Loss: 0.6099
Epoch 060 | Binary Cross-Entropy Loss: 0.5739
Epoch 080 | Binary Cross-Entropy Loss: 0.5413
Epoch 100 | Binary Cross-Entropy Loss: 0.5118
Epoch 120 | Binary Cross-Entropy Loss: 0.4849
Epoch 140 | Binary Cross-Entropy Loss: 0.4604
Epoch 160 | Binary Cross-Entropy Loss: 0.4382
Epoch 180 | Binary Cross-Entropy Loss: 0.4178


Prediction Function

In [12]:
def predict(X: np.ndarray, W: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Predicts the binary class (0 or 1) for given input features using trained weights.
    
    Args:
        X (np.ndarray): Input feature matrix
        W (np.ndarray): Trained weight vector
        b (np.ndarray): Trained bias
        
    Returns:
        np.ndarray: Binary predictions (0 or 1)
    """
    # Compute probabilities using forward pass
    Z: np.ndarray = np.dot(X, W) + b
    A: np.ndarray = sigmoid(Z)

    # Threshold the probability at 0.5 to declare Anomaly (1) or Normal (0)
    # .astype(int) converts boolean True/False into 1/0
    return (A > 0.5).astype(int)


Evaluate Neural Network

In [13]:
# Make predictions using the Custom Neural Network
y_pred_nn: np.ndarray = predict(X_test, W, b)

# Measure its accuracy compared to the actual test labels
accuracy_nn: float = accuracy_score(y_test, y_pred_nn)

print(f"Custom Neural Network Accuracy: {accuracy_nn:.4f}")

Custom Neural Network Accuracy: 0.9797


What is self-learning in this case?

Neural networks are considered “self-learning” because they automatically adjust their internal parameters (weights and biases) using data. During training, the model makes predictions, calculates the error between predicted and true values, and then uses the backpropagation algorithm to compute how each weight contributed to the error. The weights are then updated using gradient descent to reduce the error. Repeating this process many times allows the model to gradually improve its predictions without manual rule programming.
